# Gold: patient_360

Builds `<catalog>.gold.patient_360` (one row per patient) from the silver layer per the STTM `sttm_silver_to_gold.xlsx`. Driving table `silver.patients`, LEFT JOINed to hospitals, insurance providers and per-patient lab result aggregates. Full rebuild via CREATE OR REPLACE TABLE AS SELECT.

**Safety:** defaults to `run_mode = preview` (build + validate, nothing written). Execution stops if any source schema (incl. `lab_results`) does not match the STTM. Writing requires `run_mode = write`, and replacing an existing table additionally requires `allow_replace_existing = yes`.

In [0]:
import re
from pyspark.sql import functions as F

# run_mode = preview -> build and validate only, nothing is written (default)
# run_mode = write   -> CREATE OR REPLACE the Gold table (only after approval)
dbutils.widgets.text("catalog", "databrickscaresync")
dbutils.widgets.dropdown("run_mode", "preview", ["preview", "write"])
dbutils.widgets.dropdown("allow_replace_existing", "no", ["no", "yes"])

catalog = dbutils.widgets.get("catalog").strip()
run_mode = dbutils.widgets.get("run_mode")
allow_replace_existing = dbutils.widgets.get("allow_replace_existing") == "yes"

# Guard against injection through the catalog widget
if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: '{catalog}'")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"
source_tables = ["patients", "hospitals", "insurance_providers", "lab_results"]

print(f"[SOURCE] {source_schema}  ->  [TARGET] {target_table}")
print(f"[RUN MODE] {run_mode} | allow_replace_existing={allow_replace_existing}")

In [0]:
schema_rows = []
for t in source_tables:
    fn = f"{source_schema}.{t}"
    if not spark.catalog.tableExists(fn):
        schema_rows.append((t, None, "<TABLE NOT FOUND>", None))
        continue
    df = spark.table(fn)
    n = df.count()
    for pos, f in enumerate(df.schema.fields, start=1):
        schema_rows.append((t, pos, f.name, f.dataType.simpleString()))
    print(f"{fn}: {n} rows, {len(df.columns)} columns")

display(spark.createDataFrame(
    schema_rows, "source_table string, position int, column_name string, data_type string"
))

In [0]:
# Source of truth: gold/sttm_silver_to_gold.xlsx, sheet 'patient_360'
sttm_mapping = [
    ("patient_id", "int", "patients.patient_id", "Pass-through (primary key)"),
    ("first_name", "string", "patients.first_name", "Pass-through"),
    ("last_name", "string", "patients.last_name", "Pass-through"),
    ("date_of_birth", "date", "patients.date_of_birth", "Pass-through"),
    ("age", "int", "patients.date_of_birth", "datediff(current_date(), date_of_birth) / 365.25, floored to int"),
    ("gender", "string", "patients.gender", "Pass-through"),
    ("city", "string", "patients.city", "Pass-through"),
    ("state", "string", "patients.state", "Pass-through"),
    ("hospital_id", "int", "patients.registered_hospital_id", "Pass-through (join key)"),
    ("hospital_name", "string", "hospitals.hospital_name", "Via LEFT JOIN on registered_hospital_id = hospital_id"),
    ("insurance_provider_id", "int", "patients.insurance_provider_id", "Pass-through (join key)"),
    ("insurance_provider_name", "string", "insurance_providers.provider_name", "Via LEFT JOIN on insurance_provider_id"),
    ("total_tests", "int", "lab_results.lab_result_id", "COUNT(*) per patient_id, 0 if none"),
    ("abnormal_test_count", "int", "lab_results.result_status", "COUNT(*) WHERE result_status = 'Abnormal' per patient_id, 0 if none"),
    ("critical_test_count", "int", "lab_results.result_status", "COUNT(*) WHERE result_status = 'Critical' per patient_id, 0 if none"),
    ("last_test_date", "date", "lab_results.test_date", "MAX(test_date) per patient_id, NULL if none"),
]
expected_types = {c: t for c, t, _, _ in sttm_mapping}

print("Grain: one row per patient | Driving table: silver.patients | Filters: none | Load: FULL (CREATE OR REPLACE)")
display(spark.createDataFrame(
    sttm_mapping, "target_column string, data_type string, source string, transformation_rule string"
))

In [0]:
required_columns = {
    "patients": ["patient_id", "first_name", "last_name", "date_of_birth", "gender",
                 "city", "state", "registered_hospital_id", "insurance_provider_id"],
    "hospitals": ["hospital_id", "hospital_name"],
    "insurance_providers": ["insurance_provider_id", "provider_name"],
    "lab_results": ["lab_result_id", "patient_id", "result_status", "test_date"],
}

schema_status, blocking = [], []
for t, cols in required_columns.items():
    fn = f"{source_schema}.{t}"
    if not spark.catalog.tableExists(fn):
        schema_status.append((fn, "FAIL", "Table not found"))
        blocking.append(fn)
        continue
    actual = [c.lower() for c in spark.table(fn).columns]
    missing = [c for c in cols if c not in actual]
    if missing:
        schema_status.append((fn, "FAIL", f"Missing {missing}; actual columns {actual}"))
        blocking.append(fn)
    elif spark.table(fn).limit(1).count() == 0:
        schema_status.append((fn, "WARN", "Schema OK but table is empty (LEFT JOIN columns will be NULL)"))
    else:
        schema_status.append((fn, "PASS", "All required columns present"))

display(spark.createDataFrame(schema_status, "source_table string, status string, detail string"))

if blocking:
    raise ValueError(
        f"[STOPPED] Source schema validation failed for {blocking}. "
        "No transformation was run and no Gold table was written. "
        "Fix the upstream Silver load and re-run."
    )
print("[VALIDATION PASSED] All source schemas match the STTM")

In [0]:
select_sql = f"""
WITH lab_agg AS (
    SELECT
        patient_id,
        COUNT(*)                             AS total_tests,
        COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical') AS critical_test_count,
        MAX(CAST(test_date AS DATE))         AS last_test_date
    FROM {source_schema}.lab_results
    GROUP BY patient_id
),
-- Defensive: keep the latest row per key so dimension joins never multiply patients
hospitals_dedup AS (
    SELECT * FROM {source_schema}.hospitals
    QUALIFY ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY ingestion_timestamp DESC, updated_at DESC) = 1
),
insurance_providers_dedup AS (
    SELECT * FROM {source_schema}.insurance_providers
    QUALIFY ROW_NUMBER() OVER (PARTITION BY insurance_provider_id ORDER BY ingestion_timestamp DESC, updated_at DESC) = 1
)
SELECT
    CAST(p.patient_id AS INT)                                                   AS patient_id,
    CAST(p.first_name AS STRING)                                                AS first_name,
    CAST(p.last_name AS STRING)                                                 AS last_name,
    CAST(p.date_of_birth AS DATE)                                               AS date_of_birth,
    CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
    CAST(p.gender AS STRING)                                                    AS gender,
    CAST(p.city AS STRING)                                                      AS city,
    CAST(p.state AS STRING)                                                     AS state,
    CAST(p.registered_hospital_id AS INT)                                       AS hospital_id,
    CAST(h.hospital_name AS STRING)                                             AS hospital_name,
    CAST(p.insurance_provider_id AS INT)                                        AS insurance_provider_id,
    CAST(ip.provider_name AS STRING)                                            AS insurance_provider_name,
    CAST(COALESCE(l.total_tests, 0) AS INT)                                     AS total_tests,
    CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                             AS abnormal_test_count,
    CAST(COALESCE(l.critical_test_count, 0) AS INT)                             AS critical_test_count,
    CAST(l.last_test_date AS DATE)                                              AS last_test_date
FROM {source_schema}.patients p
LEFT JOIN hospitals_dedup h            ON p.registered_hospital_id = h.hospital_id
LEFT JOIN insurance_providers_dedup ip ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l                    ON p.patient_id = l.patient_id
"""

preview_df = spark.sql(select_sql)
preview_df.createOrReplaceTempView("patient_360_preview")

print("[PREVIEW] Transformation built as temp view patient_360_preview (nothing written)")
display(preview_df.orderBy("patient_id").limit(20))

In [0]:
checks = []

def add_check(name, passed, detail, blocking=True):
    checks.append((name, "PASS" if passed else ("FAIL" if blocking else "WARN"), str(detail)))

src_patients = spark.table(f"{source_schema}.patients").count()
m = spark.sql("""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT patient_id) AS distinct_patients,
           COUNT_IF(patient_id IS NULL) AS null_keys,
           COUNT_IF(hospital_name IS NULL) AS null_hospital_name,
           COUNT_IF(insurance_provider_name IS NULL) AS null_insurance_name,
           SUM(total_tests) AS gold_total_tests
    FROM patient_360_preview
""").first()
src_tests = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {source_schema}.lab_results l
    WHERE EXISTS (SELECT 1 FROM {source_schema}.patients p WHERE p.patient_id = l.patient_id)
""").first().n

duplicates_df = spark.sql("""
    SELECT patient_id, COUNT(*) AS row_count FROM patient_360_preview
    GROUP BY patient_id HAVING COUNT(*) > 1
""")
dup_count = duplicates_df.count()

actual_types = {f.name: f.dataType.simpleString() for f in preview_df.schema.fields}
type_mismatch = {c: (expected_types.get(c), actual_types.get(c))
                 for c in set(expected_types) | set(actual_types)
                 if expected_types.get(c) != actual_types.get(c)}

add_check("Row count = silver.patients", m.total_rows == src_patients, f"gold={m.total_rows}, silver={src_patients}")
add_check("No duplicate patient_id (grain)", dup_count == 0, f"duplicate keys={dup_count}")
add_check("No NULL patient_id", m.null_keys == 0, f"null keys={m.null_keys}")
add_check("Columns and data types match STTM", not type_mismatch, type_mismatch or "all 16 columns match")
add_check("SUM(total_tests) = lab_results rows for known patients", m.gold_total_tests == src_tests,
          f"gold={m.gold_total_tests}, silver={src_tests}")
add_check("hospital_name populated", m.null_hospital_name == 0, f"NULL hospital_name={m.null_hospital_name}", blocking=False)
add_check("insurance_provider_name populated", m.null_insurance_name == 0,
          f"NULL insurance_provider_name={m.null_insurance_name}", blocking=False)

print(f"Row counts -> silver.patients: {src_patients} | preview: {m.total_rows}")
print("Duplicate keys:")
display(duplicates_df)
display(spark.createDataFrame(checks, "check string, status string, detail string"))

validation_passed = all(s != "FAIL" for _, s, _ in checks)
print("[VALIDATION STATUS]", "PASSED" if validation_passed else "FAILED")

In [0]:
if run_mode != "write":
    print(f"[PREVIEW MODE] Nothing written. {target_table} untouched. "
          "Set run_mode=write after approval to load it.")
else:
    if not validation_passed:
        raise ValueError("[STOPPED] Validation failed; Gold table not written.")
    if spark.catalog.tableExists(target_table) and not allow_replace_existing:
        raise ValueError(f"[STOPPED] {target_table} already exists. "
                         "Set allow_replace_existing=yes to replace it.")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")
    spark.sql(f"CREATE OR REPLACE TABLE {target_table} AS {select_sql}")
    print(f"[DONE] Rebuilt {target_table}: {spark.table(target_table).count()} rows")

In [0]:
# Inspect the current Bronze table mappings
for table_name in [
    "patients",
    "hospitals",
    "doctors",
    "insurance_providers",
    "lab_results"
]:
    table = f"databrickscaresync.bronze.{table_name}"
    print(f"\n========== {table_name} ==========")
    print(spark.table(table).columns)
    print("Rows:", spark.table(table).count())

In [0]:
updated_df = spark.table("databrickscaresync.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str)